# Types

## Introduction

Julia type system is **dynamic**, **nominative**, and **parametric**.
  * **dynamic**: Nothing is known about the types until runtime, when the actual values manipulated by the program are available. However, type annotation is built into the language which enables efficient code generation (compilation?) and multiple dispatch methods.
  * [**nominative**](https://en.wikipedia.org/wiki/Nominal_type_system): As opposed to [structural type system](https://en.wikipedia.org/wiki/Structural_type_system), nominative types mean that just because two types have the same "structure" (same properties, same names, same methods, etc.) does not mean that they are the same types, they *must* have the same name. This is pretty much the type system in all the programming languages I know. Nothing to see here.
  * **parametric**: This just means that I can have generic types.

In Julia, concrete types cannot subtype each other, all concrete types are "final". They can have multiple abstract types in their heirarchy though.

Julia has a concept of a "top" type like most other languages called `Any`. All other types are subtypes of this. It is the single root node. It also has the concept of a "bottom" type called `Union{}`. No type is a subtype of this. All types are its supertypes. This is the only leaf node. This is different from other languages where the type tree has multiple leaves in the form of final concrete types. In Julia, all those concrete types are not leaves, they all have an edge to the `Union{}` type.

Julia has algebraic data types - sum types as `Union` and product types as **Composite** types. 

## `::` Operator

The `::` operator is used in two similar but slightly different ways -
  * As an in-place type assertion for expressions: This is a bit of an unsual feature. Any arbitrarily complex expression can be put in parens and have the expected type specified. If the expression evaluates to that type well and good, otherwise a `TypeError` will be thrown. 
  * As a type annotation: This is like most other languages, it used during assignment. Julia will use the `convert` function to safely convert the RHS to the type specified on the LHS. If this fails, it will throw an `InexactError`.

In [1]:
# As a type annotation
# convert(Int, 3.0) will safely convert 3.0 to 3
x::Int = 3.0

3.0

In [2]:
# but convert(Int, 3.1) will fail
y::Int = 3.1

InexactError: InexactError: Int64(3.1)

In [3]:
# As a type assertion
x = 1
y = 3
(x^2 + x * y + y^2)::Int

13

In [4]:
# This fails
(x^3 + x * y^2 + x^2 * y + y^3)::AbstractFloat

TypeError: TypeError: in typeassert, expected AbstractFloat, got a value of type Int64

## Sum Types aka Unions

In Julia sum types can be declared as `Union`s. It is useful when fields of structs need to be of mixed type, like `Nothing` or an `Int`. For function parameters it is not very useful because I can simply define multiple methods, one for each type in the union. It can be useful to specify `Maybe` style return types, but then in Julia we don't usually specify return types.

In [5]:
struct User
    id::Union{String,Int}
    name::String
end

In [6]:
User("APTG", "Avilay")

User("APTG", "Avilay")

In [7]:
User(1, "Anu")

User(1, "Anu")

In [8]:
MaybeInt = Union{Nothing,Int}

Union{Nothing, Int64}

In [9]:
function f(x)::MaybeInt
    if rand() > 0.5
        x + 1
    else
        nothing
    end
end

f (generic function with 1 method)

In [10]:
for _ in 1:10
    v = f(1)
    println(typeof(v), " ", v)
end

Nothing nothing
Nothing nothing
Int64 2
Int64 2
Nothing nothing
Int64 2
Nothing nothing
Int64 2
Nothing nothing
Int64 2


As can be seen from above, the output is not really a new type called `MaybeInt`, it is either `Nothing` or an `Int64`.

In [11]:
function g(x::MaybeInt)
    println(typeof(x))
    if isnothing(x)
        1
    else
        x + 1
    end
end

g (generic function with 1 method)

In [12]:
g(1)

Int64


2

In [13]:
g(nothing)

Nothing


1

As can be seen from above, the input is also not a brand new type called `MaybeInt`, it is either `Nothing` or an `Int64`.

In [14]:
# A much better implementation
g_(x::Nothing) = 1
g_(x::Int) = x + 1

g_ (generic function with 2 methods)

In [15]:
g_(nothing)

1

In [16]:
g_(1)

2

## Product Types aka Composite Types

  * Properties can be type annotated or not, if not then defaults to `Any` as usual.
  * Two default ctors - one that takes in the exact types of properties, another that takes in any types and converts them appropriately if possible.
  * `===` between two different objects but with same properties will evaluate to true.
  * Get a list of defined properties using `fieldnames()` function.
  * Are immutable, so cannot change the value of a field/property once it is set during construction time.
  * The values themselves can be mutable types like vectors, which can be changed, just the property cannot point to another object.
  * Can be made mutable.
  * Mutable structs can have some immutable fields.

Because of multiple dispatch, there is no notion of object methods in Julia. A function is dispatched based on the types of **all** its input parameters, a traditional OO method would require that the dispatch be based just on the first parameter (self or this).

In [17]:
struct Cookie
    flavor::String
    calories::Int
end

In [18]:
c1 = Cookie("Chocolate Chip", 200)
c2 = Cookie("Chocolate Chip", 200)

Cookie("Chocolate Chip", 200)

In [19]:
c1 == c2

true

In [20]:
c1 === c2

true

In [21]:
struct Pair
    x
    y
end

In [22]:
p1 = Pair(1, "hello")
p2 = Pair(3.14, 2.71)

Pair(3.14, 2.71)

In [23]:
# calories will be converted to Int
c3 = Cookie("Snicker Doodle", 220.0)

Cookie("Snicker Doodle", 220)

In [24]:
fieldnames(Cookie)

(:flavor, :calories)

In [25]:
println(c3.flavor, " ", c3.calories)

Snicker Doodle 220


In [26]:
# The object is immuatable, this will fail.
c3.flavor = "Oatmeal Raisin"

ErrorException: setfield!: immutable struct of type Cookie cannot be changed

In [27]:
struct CookieJar
    cookies::Vector{Cookie}
    capacity::Int
end

In [28]:
jar = CookieJar([Cookie("Chocolate Chip", 200), Cookie("Chocolate Chip", 200)], 5)

CookieJar(Cookie[Cookie("Chocolate Chip", 200), Cookie("Chocolate Chip", 200)], 5)

In [29]:
jar.cookies

2-element Vector{Cookie}:
 Cookie("Chocolate Chip", 200)
 Cookie("Chocolate Chip", 200)

In [30]:
# But the contained object can be mutated if it is a mutable object
push!(jar.cookies, Cookie("Oatmeal Raisin", 180))

3-element Vector{Cookie}:
 Cookie("Chocolate Chip", 200)
 Cookie("Chocolate Chip", 200)
 Cookie("Oatmeal Raisin", 180)

In [31]:
# This will fail because the jar is still immutable
jar.capacity = 10

ErrorException: setfield!: immutable struct of type CookieJar cannot be changed

In [32]:
mutable struct Vector2D
    x::AbstractFloat
    y::AbstractFloat
    const tag::String
end

In [33]:
v1 = Vector2D(1, 1, "i")
v2 = Vector2D(1, 1, "i")

Vector2D(1.0, 1.0, "i")

In [34]:
v1 == v2

false

In [35]:
v1 === v2

false

In [36]:
v3 = v1

Vector2D(1.0, 1.0, "i")

In [37]:
v1 == v3

true

In [38]:
v1 === v3

true

In [39]:
v1.x = 0
v1

Vector2D(0.0, 1.0, "i")

In [40]:
# While Vector2D is mutable, the tag field has been marked as const and is immutable.
# This will fail
v1.tag = "j"

ErrorException: setfield!: const field .tag of type Vector2D cannot be changed

### Constructors

Ref: https://docs.julialang.org/en/v1/manual/constructors/

There are a lot of nuances with constructors. In this notebook I'll cover the basic usage. For details refer to the manual.

There are two kinds of constructors -
  * Outer constructors
  * Inner constructors

Outer constructors are simply eponymously named functions with different parameters where multiple dispatch does its magic. They will eventually need to call inner constructor to actually create the object. The default constructors are inner constructors. Both default constructors have the same number of parameters as the number of fields. But I can create my own inner constructor. Once I define even a single new inner constructor, Julia will not generate the default inner constructors for me.

Inner constructors call the special `new()` function to create the object. This function can also create partially initialized object. I have not given an example of that in this notebook.

In [ ]:
# A couple of outer constructors

function Cookie(flavor::String)
    if flavor == "Chocolate Chip"
        Cookie(flavor, 200)
    elseif flavor == "Snicker Doodle"
        Cookie(flavor, 220)
    elseif flavor == "Oatmeal Raisin"
        Cookie(flavor, 180)
    else
        Cookie(flavor, 250)
    end
end

function Cookie()
    Cookie("Chocolate Chip", 200)
end

Cookie

In [44]:
Cookie("Oatmeal Raisin")

Cookie("Oatmeal Raisin", 180)

In [45]:
Cookie()

Cookie("Chocolate Chip", 200)

In [46]:
# I still have the default ctors
Cookie("Gluten Free Chocolate Chip", 150)

Cookie("Gluten Free Chocolate Chip", 150)

In [47]:
struct Book
    id::Int
    title::String

    function Book(title::String)
        # Create a new book in the db
        dbid = rand(1:100)

        # Call the special new function
        new(dbid, title)
    end
end

In [48]:
Book("Lord of the Rings")

Book(85, "Lord of the Rings")

In [ ]:
# I lost the default constructor so this call will fail
Book(1, "Sherlock Holmes")

MethodError: MethodError: no method matching Book(::Int64, ::String)
The type `Book` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  Book(!Matched::String)
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y351sZmlsZQ==.jl:5


### Functors
The term "functor" here refers to a callable object (a struct instance that can be called like a function). This is Julia's convention, borrowed from C++. It's unrelated to the category theory / Haskell meaning of Functor (a mappable type constructor).

You define a struct, then add a method to the call operator for that struct type.

In [1]:
struct Modulo
    n::Integer
end

(m::Modulo)(x) = x % m.n

In [2]:
mod3 = Modulo(3)

Modulo(3)

Now `mod3` is a "callable", I can call it with the parameter `x`.

In [3]:
mod3(10)

1

**Why use this over a lambda or a "closure"?**

I could've defined the following simple lambda that did the same thing -

```julia
make_mod(n) = x -> x % n
mod3 = make_mod(3)
```

Here are the reasons a functor differs from this -

  * I can inspect the parameter `n` at any time with `mod3.n` with a functor, not so in lambda.
  * I can use multiple dispatch when passing functors to functions because each functor is its own type. In Julia it is not easy (or even possible?) to annotate a function with its exact signature, so all functions would just be `Function` or `Base.Callable`.

## Style Guide

  * Prefer exported methods over direct field access.
  * Non-exported functions are typically internal and can be given a `_` prefix.
  * Use `isa` and `<:` for testing types, not `==`.

## Pretty Printing

I need to add new `Base.show` methods for customizing the printing.

  * Override `Base.show(io::IO, cookie::Cookie)` for basic text printing.
  * Override `Base.show(io::IO, ::MIME"text/html", cookie::Cookie)` for printing to html.

Just like HTML, there are a bunch of different MIME types that I can customize the printing for. The Jupyter notebook will typically call the `display(obj)` method which will then call the appropriate `show` method. Calling `print(obj)` directly will call the `show(stdout, obj)` method.

In [50]:
function Base.show(io::IO, cookie::Cookie)
  print(io, "🍪 $(cookie.flavor) has $(cookie.calories) calories")
end

function Base.show(io::IO, ::MIME"text/html", cookie::Cookie)
  str = """<strong>Cookie</strong>
<ul>
  <li>$(cookie.flavor)</li>
  <li>$(cookie.calories) calories</li>
</ul>"""
  println(io, str)
end

In [51]:
# This will call display(c1)
c1

🍪 Chocolate Chip has 200 calories

In [52]:
print(c1)

🍪 Chocolate Chip has 200 calories

## Singleton Types

If I have a struct with no fields, it is a singleton type, there can only ever be a single instance of it. `Nothing` is such a type and `nothing` is such an instance.

In [53]:
struct Neo
end

In [54]:
theone = Neo()

Neo()

In [55]:
thetwo = Neo()

Neo()

In [56]:
println(theone == thetwo)
println(theone === thetwo)

true
true


In [57]:
Base.issingletontype(Neo)

true

In [58]:
Base.issingletontype(Cookie)

false

## Abstract Types

This is a bit unusual. Unlike other languages, abstract types in Julia are just names or tags. They don't have any properties. What is even more unusual is that there is no formal concept of an interface either! There are no functions/methods that concrete subtypes of a given abstract type must implement. However, the concept of interfaces is very much alive in the language. These are specified as part of docstrings. The main benefit of abstract types is that it lets me define a function that can operate on all different subtypes. This is best shown with examples.

In [59]:
"""
    Shape
A type to represent geometric shapes.

## Interface
All concrete shapes must implement the following functions -
  * area :: Shape -> Number
"""
abstract type Shape end

Shape

Now, I can define a function that calculates the volume of 3D prisms with different types of shapes as bases.

In [60]:
function prism_volume(shape::Shape, height::Number)
    area(shape) * height
end

prism_volume (generic function with 1 method)

This function assumes that there is a method called `area` defined for all concrete subtypes of `Shape`. It uses that to do other things with the shape.

A concrete type implementing `Shape` uses the `<:` keyword/operator(?) to specify its parent. The `<:` operator also doubles as the `is subclass of` function.

In [61]:
struct Circle <: Shape
    radius::Number
end

area(c::Circle) = π * c.radius^2

area (generic function with 1 method)

In [62]:
c = Circle(2)
area(c)

12.566370614359172

In [63]:
Circle <: Shape

true

In [64]:
Cookie <: Shape

false

In [65]:
struct Square <: Shape
    side::Number
end

area(s::Square) = s.side^2

area (generic function with 2 methods)

In [66]:
s = Square(2)
area(s)

4

In [67]:
prism_volume(s, 2)

8

In [68]:
prism_volume(c, 2)

25.132741228718345

Without the abstract `Shape`, I'd have to define multiple methods of `prism_volume`, one for each concrete shape. Of course I still have to define certain "primitives" like `area` for each concrete shape. But the idea is that I can build several more complex functions like `prism_volume` which can combine these primitives in different ways for more complex functionality.

Now, lets say I define a new shape - the triangle, but forget to implement its area.

In [69]:
struct Triangle <: Shape
    a::Number
    b::Number
    c::Number
end

In [70]:
t = Triangle(3, 4, 5)

Triangle(3, 4, 5)

In [71]:
# This will fail because area is not defined
prism_volume(t, 2)

MethodError: MethodError: no method matching area(::Triangle)
The function `area` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  area(!Matched::Square)
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y134sZmlsZQ==.jl:5
  area(!Matched::Circle)
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y130sZmlsZQ==.jl:5


I get a `MethodError` on `area` which is a good enough indication that I forgot something. However, in Julia we often use the so-called "Loud Failure" pattern, where I implement the `area` method for `Shape` itself which will raise an error with a more user-friendly message.

In [72]:
area(s::Shape) = error("You must implement the `area :: $(typeof(s)) -> Number` method for $(typeof(s))!")

area (generic function with 3 methods)

In [73]:
# Fail but with better error message
prism_volume(t, 2)

ErrorException: You must implement the `area :: Triangle -> Number` method for Triangle!

Here is the hierarchy of numeric types -

```
Number (abstract)
├── Complex{T<:Real}
└── Real (abstract)
    ├── AbstractFloat (abstract)
    │   ├── Float16
    │   ├── Float32
    │   ├── Float64
    │   └── BigFloat
    ├── AbstractIrrational (abstract)
    │   └── Irrational{sym}
    ├── Integer (abstract)
    │   ├── Bool
    │   ├── Signed (abstract)
    │   │   ├── Int8
    │   │   ├── Int16
    │   │   ├── Int32
    │   │   ├── Int64
    │   │   ├── Int128
    │   │   └── BigInt
    │   └── Unsigned (abstract)
    │       ├── UInt8
    │       ├── UInt16
    │       ├── UInt32
    │       ├── UInt64
    │       └── UInt128
    └── Rational{T<:Integer}
```

`Int` and `UInt` are just platform independent aliases for platform specific `Int64` and `UInt64` on 64-bit systems.

## Declared Types

Just like in Haskell we have the concept of a concrete type `*` which is the type of the data type themselves, similarly in Julia we have the concept of a `DataType` which is the type of the type itself. Unlike Haskell, this is not just limited to concrete aka composite types. Abstract types and primitive types (not covered in this notebook) are also of type `DataType`. Of course Haskell has no notion of abstract or primitive types so the analogy breaks down there. Any type which I have to explicitly declare will be a `DataType`. I can contrast this type with the `UnionAll` type covered later in the notebook which is not explicitly declared but comes into existence when I declare parametric types. I can declare type aliases of `UnionAll` types which are themselves `UnionAll` types so it might seem like I am declaring a `UnionAll` type, but remember it is only an alias. I cannot explicitly declare a `UnionAll` type.

In [74]:
println(typeof(Shape))
println(typeof(Square))

DataType
DataType


## Parametric Types

Unlike regular composite types, there is only one default constructor generated for parametric types. This takes in any type of inputs and attempts to convert them. But this constructor can be called in two different ways - with specifying the type parameter, and without specifying the type parameter. In the latter case, Julia will use type inference.

In [75]:
struct Node{T}
    data::T
    next::Union{Nothing,Node{T}}
end

In [76]:
# Type is explicitly specified
one = Node{Int}(1, nothing)

Node{Int64}(1, nothing)

In [77]:
# Type is automatically inferred
two = Node(2, one)

Node{Int64}(2, Node{Int64}(1, nothing))

In [78]:
# Type is converted
Node{Float64}(1, nothing)

Node{Float64}(1.0, nothing)

In [79]:
# This will not work, because the entire Node hierarchy needs to be of the same type
three = Node{String}("three", two)

MethodError: MethodError: Cannot `convert` an object of type 
  Node{Int64} to an object of type 
  Node{String}
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  (::Type{Node{T}} where T)(::Any, !Matched::Any)
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y160sZmlsZQ==.jl:2
  convert(::Type{T}, !Matched::T) where T
   @ Base Base_compiler.jl:133


I can mix integers and floats by explicitly setting the type to `Real` in the constructor.

In [80]:
Node{Real}(
    1,
    Node{Real}(
        1.1,
        nothing
    )
)

Node{Real}(1, Node{Real}(1.1, nothing))

In Julia, like in most other languages, parametric types are invariant.

In [81]:
Float64 <: Real

true

In [82]:
Node{Float64} <: Node{Real}

false

In [83]:
Node{Real} <: Node{Float64}

false

Practically speaking this means that I cannot call a function that takes in `Node{Real}` with `Node{Float64}` or vice-versa. 

In [84]:
function maxnode(root::Node{Real})
    currmax = root
    curr = root
    while !isnothing(curr)
        if curr.data > currmax.data
            currmax = curr
        end
        curr = curr.next
    end
    currmax
end

function minnode(root::Node{Float64})
    currmin = root
    curr = root
    while !isnothing(curr)
        if curr.data < currmin.data
            currmin = curr
        end
        curr = curr.next
    end
    currmin
end

minnode (generic function with 1 method)

In [85]:
realnode = Node{Real}(
    2.2,
    Node{Real}(
        4.4,
        Node{Real}(
            3.3,
            nothing
        )
    )
)

floatnode = Node{Float64}(
    2.2,
    Node{Float64}(
        4.4,
        Node{Float64}(
            3.3,
            nothing
        )
    )
)

Node{Float64}(2.2, Node{Float64}(4.4, Node{Float64}(3.3, nothing)))

In [86]:
maxnode(realnode)

Node{Real}(4.4, Node{Real}(3.3, nothing))

In [87]:
try
    maxnode(floatnode)
catch e
    @assert e isa MethodError
    println("💣 Got MethodError!")
end

💣 Got MethodError!


In [88]:
try
    minnode(realnode)
catch e
    @assert e isa MethodError
    println("💣 Got MethodError!")
end

💣 Got MethodError!


In [89]:
minnode(floatnode)

Node{Float64}(2.2, Node{Float64}(4.4, Node{Float64}(3.3, nothing)))

Functions like this are best constrained with type bounds instead of tying them down to a specific type like `Real` or `Float64`.

In [90]:
function sumnodes(root::Node{<:Real})
    tot = 0
    curr = root
    while !isnothing(curr)
        tot += curr.data
        curr = curr.next
    end
    tot
end

sumnodes (generic function with 1 method)

In [91]:
sumnodes(floatnode)

9.9

In [92]:
sumnodes(realnode)

9.9

I can create nodes of any type, not just numbers. As long as the entire hierarchy is nodes of the same type, I am good.

In [93]:
cn1 = Node(Cookie("Chocolate Chip", 200), nothing)
cn2 = Node(Cookie("Snicker Doodle", 220), cn1)

Node{Cookie}(🍪 Snicker Doodle has 220 calories, Node{Cookie}(🍪 Chocolate Chip has 200 calories, nothing))

I can also constraint the type of nodes I can create. In the example below, I can create numeric nodes, but not any other type of nodes. Also a common mistake is to think that I can mix and match numeric types, but that is not true. I cannot have an int node pointing to a float node!

In [94]:
struct NumericNode{T<:Number}
    data::T
    next::Union{NumericNode{T},Nothing}
end

In [95]:
# Everything works because its all of NumericNode{Int} type.
nn1 = NumericNode(1, nothing)
nn2 = NumericNode(2, nn1)

NumericNode{Int64}(2, NumericNode{Int64}(1, nothing))

In [96]:
nn3 = NumericNode(1.1, nothing)
nn4 = NumericNode(2.2, nn3)

NumericNode{Float64}(2.2, NumericNode{Float64}(1.1, nothing))

In [97]:
# Of course non-numeric types won't work at all
NumericNode("a", nothing)

MethodError: MethodError: no method matching NumericNode(::String, ::Nothing)
The type `NumericNode` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  NumericNode(!Matched::T, ::Union{Nothing, NumericNode{T}}) where T<:Number
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y224sZmlsZQ==.jl:2


In [98]:
# If I try to mix and match numeric types, that is not going to work either.
nn5 = NumericNode(1, nothing)  # this will be NumericNode{Int}
nn6 = NumericNode(1.1, nn5)  # this will be NumericNode{Float64}

MethodError: MethodError: no method matching NumericNode(::Float64, ::NumericNode{Int64})
The type `NumericNode` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  NumericNode(::T, !Matched::Union{Nothing, NumericNode{T}}) where T<:Number
   @ Main ~/projects/github/learn-langs/learn-julia/jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_Y224sZmlsZQ==.jl:2


I can of course explicitly use the `NumericNode{Real}` constructor for this as before. Because `Real` is a subtype of `Number`, that will work.

In [100]:
NumericNode{Real}(
    1,
    NumericNode{Real}(
        1.1,
        nothing
    )
)

NumericNode{Real}(1, NumericNode{Real}(1.1, nothing))

## `UnionAll`

In the examples above `Node{Int}`, `Node{String}`, etc. are concrete subtypes of `Node`. `Node` itself is of a weird type called `UnionAll`. The best way I have been able to internalize this type is as akin to type constructor in Haskell. `Node` can be more accurately written as `Node{T} where T`, which means that `Node` is the union of all possible types `T`.

In [101]:
Node{Int} <: Node

true

In [102]:
typeof(Node{Int})

DataType

In [103]:
typeof(Node)

UnionAll

#### Type Alias

Using type alias I can create a concrete type aka a `DataType` or a type constructor aka `UnionAll` type.

In [104]:
const IntNode = Node{Int}

IntNode (alias for Node{Int64})

In [105]:
typeof(IntNode)

DataType

In [106]:
n1::IntNode = Node(1, nothing)

IntNode(1, nothing)

In [107]:
typeof(n1)

IntNode (alias for Node{Int64})

In [108]:
n2::IntNode = Node(2, n1)

IntNode(2, IntNode(1, nothing))

In [109]:
const StringNode = Node{String}

StringNode (alias for Node{String})

In [110]:
# Here even though I am not specifying the type, Julia's type inference automatically picks up that
# these should be StringNodes
s1 = Node("a", nothing)
s2 = Node("b", s1)

StringNode("b", StringNode("a", nothing))

It seems like I can define type alias with type constraints in three different ways -

  * The aliased type does not have the type parameter
  ```julia
  const RealNode = Node{T} where T<:Real
  ```

  * Shorthand for when the type alias does not have the type parameter
  ```julia
  const RealNode = Node{<:Real}
  ```

  * The aliased type does have the type parameter
  ```julia
  const RealNode{T} = Node{T} where T<:Real
  ```

All are `UnionAll` types. According to the manual the first two ways are equivalent, the second is just syntactic sugar for the first. The third way is not mentioned. All seem to work exactly like each other, but I have not been able to tease out the difference between how the ones without type and one with typee work. Maybe they are all equivalent, and the "correct" way to declare is without the `T`.

Further on in the section on [`UnionAll` in the Manual](https://docs.julialang.org/en/v1/manual/types/#UnionAll-Types) it says that `const Vector = Array{T,1} where T` and `Vector{T} = Array{T,1}` are equivalent, the latter being shorthand for the former.

In [111]:
const RealNode1 = Node{T} where T<:Real

RealNode1 (alias for Node{T} where T<:Real)

In [112]:
typeof(RealNode1)

UnionAll

In [113]:
const RealNode2{T} = Node{T} where T<:Real

Node{T} where T<:Real

In [114]:
typeof(RealNode2)

UnionAll

In [115]:
const RealNode3 = Node{<:Real}

Node{<:Real}

In [116]:
typeof(RealNode3)

UnionAll

In [117]:
r1::RealNode1 = Node(1, nothing)

Node{Int64}(1, nothing)

In [118]:
r2::RealNode2 = Node(1, nothing)

Node{Int64}(1, nothing)

In [119]:
r3::RealNode1{Real} = Node{Real}(1, nothing)

Node{Real}(1, nothing)

In [120]:
r4::RealNode2{Real} = Node{Real}(1, nothing)

Node{Real}(1, nothing)

In [121]:
println(typeof(r1), " ", typeof(r1.data))
println(typeof(r2), " ", typeof(r2.data))
println(typeof(r3), " ", typeof(r3.data))
println(typeof(r4), " ", typeof(r4.data))

Node{Int64} Int64
Node{Int64} Int64
Node{Real} Int64
Node{Real} Int64


Here are some more sophisticated examples of using type constraints. The array type is defined with two parameters - `Array{T,N}`, where `T` is the type of element and `N` is the number of dimensions (not the size as I keep mistaking).

> I haven't figured out how to create types with "value" parameters like N yet.

Also remember that `Array{T, 1}` is an alias for `Vector{T}`.

In [122]:
# This gives me a vector of vectors, but each vector can have different types because the where is
# inside
const T1 = Array{Array{T,1} where T,1}

Vector{Vector} (alias for Array{Array{T, 1} where T, 1})

In [123]:
t1::T1 = [
    [Cookie("Chocolate Chip", 200), Cookie("Snicker Doodle", 220)],
    [1.1, 2.2, 3.3]
]

2-element Vector{Vector}:
 Cookie[🍪 Chocolate Chip has 200 calories, 🍪 Snicker Doodle has 220 calories]
 [1.1, 2.2, 3.3]

In [124]:
# This also gives me a vector of vectors, all of which have the same type T because the where is outside
# so it is specifying the type for all the nested arrays
const T2 = Array{Array{T,1},1} where T

Array{Vector{T}, 1} where T

In [125]:
vv::T2 = [
    [1, 2, 3],
    [4, 5, 6]
]

2-element Vector{Vector{Int64}}:
 [1, 2, 3]
 [4, 5, 6]

In [126]:
# This will fail because I cannot mix and match types
t2::T2 = [
    [Cookie("Chocolate Chip", 200), Cookie("Snicker Doodle", 220)],
    [1.1, 2.2, 3.3]
]

MethodError: MethodError: no method matching (Array{Vector{T}, 1} where T)(::Vector{Vector})
The type `Array{Vector{T}, 1} where T` exists, but no method is defined for this combination of argument types when trying to construct it.

Now here is the unexpected part! In `T1` the type does not really matter, any element in any vector can be of any type. So it is not parametric at all, it is a concrete `DataType`! `T2` on the other hand still needs to be specified for different types, so it remains a `UnionAll`.

In [127]:
println(typeof(T1))
println(typeof(T2))

DataType
UnionAll


I can define another type alias similar to `Vector` in two different ways.

In [128]:
Image{T} = Array{T,3}

Array{T, 3} where T

In [129]:
typeof(Image)

UnionAll

In [130]:
img::Image = rand(3, 5, 5)

3×5×5 Array{Float64, 3}:
[:, :, 1] =
 0.270481  0.503572  0.719422  0.79723   0.352726
 0.319688  0.236832  0.850312  0.870737  0.576125
 0.413175  0.800504  0.726874  0.606505  0.783866

[:, :, 2] =
 0.420763  0.274859  0.28749   0.809733   0.206929
 0.209005  0.496736  0.149658  0.0379424  0.415789
 0.451748  0.62562   0.959771  0.0724964  0.0146005

[:, :, 3] =
 0.0830474  0.241806   0.730575   0.880768  0.836807
 0.693551   0.715031   0.0574534  0.930548  0.16101
 0.164162   0.0166033  0.807883   0.795653  0.699238

[:, :, 4] =
 0.58208   0.941561  0.0613799  0.276953  0.858697
 0.465738  0.120696  0.741702   0.584208  0.935446
 0.124575  0.673515  0.271957   0.613786  0.155857

[:, :, 5] =
 0.957053  0.423449  0.692397  0.812676  0.200216
 0.837257  0.86626   0.289586  0.382347  0.852622
 0.80806   0.136634  0.35654   0.004235  0.69761

In [131]:
typeof(img)

Array{Float64, 3}

In [132]:
const Img = Array{T,3} where T

Array{T, 3} where T

In [133]:
typeof(Img)

UnionAll

In [134]:
img2::Img = rand(3, 5, 5)

3×5×5 Array{Float64, 3}:
[:, :, 1] =
 0.754813  0.239697  0.284327  0.497185  0.224644
 0.913241  0.312194  0.722958  0.299073  0.860411
 0.423835  0.588079  0.772278  0.765654  0.575581

[:, :, 2] =
 0.896055    0.207704  0.634978  0.291771  0.241329
 0.00715703  0.399683  0.298323  0.933931  0.155209
 0.999947    0.857805  0.871833  0.418604  0.89447

[:, :, 3] =
 0.526628  0.864338  0.930788    0.918784  0.112316
 0.707039  0.827466  0.773545    0.661508  0.235144
 0.145098  0.833993  0.00633262  0.948504  0.021006

[:, :, 4] =
 0.0122778  0.513826  0.543804  0.363047  0.29479
 0.148039   0.190365  0.608129  0.456623  0.256528
 0.622371   0.194093  0.555083  0.355815  0.701137

[:, :, 5] =
 0.808357  0.00824872  0.696254  0.294744  0.208008
 0.318586  0.201345    0.634798  0.828236  0.548461
 0.520951  0.751856    0.500342  0.386409  0.900693

In [135]:
typeof(img2)

Array{Float64, 3}

## Abstract Parameteric Types

Just like composite parametric types, I can define abstract parametric types with the same semantics. Right now I am not able to come up with a concrete real world use case for this, so documenting the syntax for future reference.

In [136]:
abstract type Pointy{T} end

In [137]:
Pointy{Int64} <: Pointy

true

In [138]:
Pointy{1} <: Pointy

true

In [139]:
println(Pointy{Float64} <: Pointy{Real})
println(Pointy{Real} <: Pointy{Float64})

false
false


In [140]:
println(Pointy{Float64} <: Pointy{<:Real})
println(Pointy{Real} <: Pointy{>:Int})

true
true


In [141]:
struct Point{T} <: Pointy{T}
    x::T
    y::T
end

In [142]:
println(Point{Float64} <: Pointy{Float64})
println(Point{Float64} <: Pointy{Real})
println(Point{Float64} <: Pointy{<:Real})

true
false
true


In [143]:
abstract type RealPointy{T<:Real} end

In [144]:
struct RealPoint{T<:Real} <: Pointy{T}
    x::T
    y::T
end